# Extracting Information
The previous tutorials assumed the data you needed to extract was provided to you either by a spreadsheet or existing API. But what do you do if the data you want either doesn't have an API or doesn't have exportable data source? Well, we would need to get and organize the data ourselves.

As an example, lets pretend that APIs don't exist for weather data (or its a paid API that we don't want to pay for).  [https://weather.com/](https://weather.com/) has all of the information we need. We have 2 options for getting this information: piggy backing off their website's API or screen scraping. Screen scraping isn't ideal and should be used as a last resort, so lets see if we can piggy back off their internal API.

## Context
Before we go into how to piggy back off a websites internal API, we need to understand how website rendering works.

Modern websites often use JavaScript to dynamically load content from APIs, which means the content you see in your browser may not be present in the initial HTML file. Webpages are rarely ever static, but there are parts of webpages that usually are static, like menus, footers, navigation bars, etc. These static elements can be hardcoded into the HTML, but the dynamic parts of the website will be rendered conditionally based on templates.

If you think about a sales page, the product listings, prices, and reviews will be loaded dynamically from an API call, so the HTML for the webpage contains placeholders that are filled in once the API response is received. The API call being made is just querying their own backend services (database) and returning that information to the front end (the UI)

## API Piggy Backing

In some cases, we can just use the results from their internal API call instead of having to screen scrape the HTML.

Lets check if we can do this:
1. Open that above URL in your preferred browser.
2. After loading the page, bring up the developer tools by right clicking the webpage and clicking on "Inspect". 
3. The "Network" tab will show you all the requests that your browser made in order to load the webpage.
4. Click on the "clear network log" button in the top left (looks like a circle with a / through it) to clear out the current requests
5. Refresh the page (not the developer tools page, but the website)
6. In the top header bar, click on "Fetch/XHR" to narrow down the results to just outgoing calls
7. Click the very first option and explore the Headers and Response tabs

### Headers
The headers section outlines the type of request, where the request was made (the url), what parameters were passed in, and other information about your browser/operating system.

### Response
The response section returns back the json for the API call that is being made. Click around in the different calls and look at their responses to see if there is anything valuable

You should see several requests being made that look valuable, such as "current", "1day", "2day", "3day", "7day", "fifteenminute", etc

So how would we piggy back off of this?

In [ ]:
# lets dissect the call being made. click on the "current" call and look at the headers tab

# we see:
# request url = https://api.weather.com/v3/wx/observations/current?geocode=39.74%2C-104.98&units=e&language=en-US&format=json&apiKey=71f92ea9dd2f4790b92ea9dd2f779061
# request method = GET

# we know fro the prior tutorials that the question marks symbolize query parameters for the request, so we can break this out into key-value pairs

params = {
    "geocode": "39.74,-104.98", # latitude and longitude of the  location, which happens to be for Denver, Colorado
    "units": "e", # units of measurement, "e" stands for English units
    "language": "en-US", # language for the response
    "format": "json", # format of the response
    "apiKey": "71f92ea9dd2f4790b92ea9dd2f779061" # API key for authentication
}

# an important thing to note is that API key is publically exposed in the request, which is not a good practice for production environments. Weather.com is slacking and we cna use that to our advantage :)

# this leaves the base URL
base_url = "https://api.weather.com/v3/wx/observations/current"

# and now lets see if we can make the request 
import requests

response = requests.get(base_url, params=params)
data = response.json()
print(data)

{'cloudCeiling': 900, 'cloudCover': 100, 'cloudCoverPhrase': 'Cloudy', 'dayOfWeek': 'Tuesday', 'dayOrNight': 'D', 'expirationTimeUtc': 1793116053, 'iconCode': 20, 'iconCodeExtend': 2000, 'obsQualifierCode': None, 'obsQualifierSeverity': None, 'precip1Hour': 0.0, 'precip6Hour': 0.0, 'precip24Hour': 0.0, 'pressureAltimeter': 30.02, 'pressureChange': 0.01, 'pressureMeanSeaLevel': 1017.9, 'pressureTendencyCode': 1, 'pressureTendencyTrend': 'Rising', 'relativeHumidity': 79, 'snow1Hour': 0.0, 'snow6Hour': 0.0, 'snow24Hour': 0.0, 'sunriseTimeLocal': '2026-10-27T07:23:26-0600', 'sunriseTimeUtc': 1793107406, 'sunsetTimeLocal': '2026-10-27T18:03:46-0600', 'sunsetTimeUtc': 1793145826, 'temperature': 44, 'temperatureChange24Hour': 1, 'temperatureDewPoint': 38, 'temperatureFeelsLike': 44, 'temperatureHeatIndex': 44, 'temperatureMax24Hour': 58, 'temperatureMaxSince7Am': 44, 'temperatureMin24Hour': 40, 'temperatureWetBulbGlobe': 44, 'temperatureWindChill': 44, 'uvDescription': 'Low', 'uvIndex': 0, 'v

In [3]:
# lets take a look at another request

#request name: fifteenminutes
# request url = https://api.weather.com/v3/wx/forecast/fifteenminute?geocode=40.76%2C-73.57&units=e&language=en-US&format=json&apiKey=71f92ea9dd2f4790b92ea9dd2f779061
# request method = GET

params = {
    "geocode": "40.76,-73.57", 
    "units": "e", 
    "language": "en-US",
    "format": "json", 
    "apiKey": "71f92ea9dd2f4790b92ea9dd2f779061" 
}

base_url = "https://api.weather.com/v3/wx/forecast/fifteenminute"
response = requests.get(base_url, params=params)
data = response.json()
print(data)

{'dayOfWeek': ['Tuesday', 'Tuesday', 'Tuesday', 'Tuesday', 'Tuesday', 'Tuesday', 'Tuesday', 'Tuesday', 'Tuesday', 'Tuesday', 'Tuesday', 'Tuesday', 'Tuesday', 'Tuesday', 'Tuesday', 'Tuesday', 'Tuesday', 'Tuesday', 'Tuesday', 'Tuesday', 'Tuesday', 'Tuesday', 'Tuesday', 'Tuesday', 'Tuesday', 'Tuesday', 'Tuesday', 'Tuesday'], 'iconCode': [32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32, 31, 31, 31, 31], 'iconCodeExtend': [3200, 3200, 3200, 3200, 3200, 3200, 3200, 3200, 3200, 3200, 3200, 3200, 3200, 3200, 3200, 3200, 3200, 3200, 3200, 3200, 3200, 3200, 3200, 3200, 3100, 3100, 3100, 3100], 'precipChance': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'precipRate': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], 'precipType': ['rain', 'rain', 'rain', 'rain', 'rain', 'rain', 'rain', 'rain', 'rain', 'rain', 'rain', '

Now that we have data, move on to tutorial #2 to see what we can do with this